# FCA Complaints Data — Cleaning & ReshapingThis notebook transforms the **Financial Conduct Authority (FCA)** firm-level complaints data from its raw Excel format into clean, long-format CSV files ready for loading into a **PostgreSQL** database.**Source:** [FCA firm-level complaints data](https://www.fca.org.uk/data/complaints-data) — downloaded as Excel workbooks for three periods: H2 2024, H1 2025, and H2 2025.**Why reshape?** The raw FCA workbooks store each product category as a separate column (*wide format*). For SQL analysis we need *long format* — one row per firm, per product, per period. This notebook performs that transformation for three metrics: complaints opened, uphold rates, and complaints closed.

## Before You Run

This notebook expects the three raw FCA Excel workbooks to be available to the notebook environment:

- `firm-level-complaints-data-2024-h2.xlsx`
- `firm-level-complaints-data-2025-h1.xlsx`
- `firm-level-complaints-data-2025-h2.xlsx`

Download these from the [FCA complaints data page](https://www.fca.org.uk/data/complaints-data) (firm-specific data), then **update the file paths in each `pd.read_excel(...)` call below** to point to wherever the files are stored on your machine or in your Kaggle/Colab input directory. Once the paths are correct, run all cells top to bottom.

## SetupImporting pandas and defining the five product categories that appear as columns in every FCA workbook.

In [ ]:
import pandas as pd# The five product categories that appear as columns in the raw FCA workbooksproduct_columns = [    "Banking and credit cards",    "Decumulation & pensions",    "Home finance",    "Insurance & pure protection",    "Investments"]

## 1. Complaints OpenedLoad the **Opened** sheet from each of the three workbooks, add a period label, combine them, then reshape from wide to long format.

In [ ]:
# Load the Opened sheet from all three filesh2_2024 = pd.read_excel("/kaggle/input/h2-2024/firm-level-complaints-data-2024-h2.xlsx",                         sheet_name="Opened", header=0)h1_2025 = pd.read_excel("/kaggle/input/h1-2025/firm-level-complaints-data-2025-h1.xlsx",                         sheet_name="Opened", header=0)h2_2025 = pd.read_excel("/kaggle/input/h2-2025/firm-level-complaints-data-2025-h2.xlsx",                         sheet_name="Opened", header=0)# Add a period label to eachh2_2024["period"] = "H2 2024"h1_2025["period"] = "H1 2025"h2_2025["period"] = "H2 2025"# Stack all three into one tabledf_opened = pd.concat([h2_2024, h1_2025, h2_2025], ignore_index=True)# Reshape wide -> long: one row per firm / product / perioddf_long = df_opened.melt(    id_vars=["Firm Name", "Group", "Reporting period", "period"],    value_vars=product_columns,    var_name="product_category",    value_name="complaints_opened")# Remove rows where the firm had no complaints in that product categorydf_long = df_long.dropna(subset=["complaints_opened"])df_long["complaints_opened"] = df_long["complaints_opened"].astype(int)# Standardise column names for SQLdf_long.columns = ["firm_name", "firm_group", "reporting_period",                   "period", "product_category", "complaints_opened"]print(f"Total rows: {len(df_long)}")df_long.head()

## 2. Uphold RatesThe uphold rate is the percentage of complaints a firm agreed were justified (i.e. the firm was at fault).**Note:** the sheet is named `Upheld` in the H2 2024 workbook but `Percentage upheld` in the 2025 workbooks, so each file is loaded individually. The FCA stores these as decimals (0.64 = 64%), so we convert to percentages.

In [ ]:
# Load the uphold sheet from each file (sheet name differs across periods)u_h2_2024 = pd.read_excel("/kaggle/input/h2-2024/firm-level-complaints-data-2024-h2.xlsx",                           sheet_name="Upheld", header=0)u_h1_2025 = pd.read_excel("/kaggle/input/h1-2025/firm-level-complaints-data-2025-h1.xlsx",                           sheet_name="Percentage upheld", header=0)u_h2_2025 = pd.read_excel("/kaggle/input/h2-2025/firm-level-complaints-data-2025-h2.xlsx",                           sheet_name="Percentage upheld", header=0)# Add period labelsu_h2_2024["period"] = "H2 2024"u_h1_2025["period"] = "H1 2025"u_h2_2025["period"] = "H2 2025"# Stack all threedf_upheld = pd.concat([u_h2_2024, u_h1_2025, u_h2_2025], ignore_index=True)# Reshape wide -> longdf_upheld_long = df_upheld.melt(    id_vars=["Firm Name", "Group", "Reporting period", "period"],    value_vars=product_columns,    var_name="product_category",    value_name="uphold_rate")# Remove empty rows and standardise column namesdf_upheld_long = df_upheld_long.dropna(subset=["uphold_rate"])df_upheld_long.columns = ["firm_name", "firm_group", "reporting_period",                          "period", "product_category", "uphold_rate"]# Convert decimals to percentages (0.64 -> 64.0)df_upheld_long["uphold_rate"] = (df_upheld_long["uphold_rate"] * 100).round(1)print(f"Total rows: {len(df_upheld_long)}")df_upheld_long.head()

## 3. Complaints ClosedSame reshaping process applied to the **Closed** sheet — complaints the firm resolved in the period.

In [ ]:
# Load the Closed sheet from all three filesc_h2_2024 = pd.read_excel("/kaggle/input/h2-2024/firm-level-complaints-data-2024-h2.xlsx",                           sheet_name="Closed", header=0)c_h1_2025 = pd.read_excel("/kaggle/input/h1-2025/firm-level-complaints-data-2025-h1.xlsx",                           sheet_name="Closed", header=0)c_h2_2025 = pd.read_excel("/kaggle/input/h2-2025/firm-level-complaints-data-2025-h2.xlsx",                           sheet_name="Closed", header=0)# Add period labelsc_h2_2024["period"] = "H2 2024"c_h1_2025["period"] = "H1 2025"c_h2_2025["period"] = "H2 2025"# Stack all threedf_closed = pd.concat([c_h2_2024, c_h1_2025, c_h2_2025], ignore_index=True)# Reshape wide -> longdf_closed_long = df_closed.melt(    id_vars=["Firm Name", "Group", "Reporting period", "period"],    value_vars=product_columns,    var_name="product_category",    value_name="complaints_closed")# Remove empty rows and standardise column namesdf_closed_long = df_closed_long.dropna(subset=["complaints_closed"])df_closed_long["complaints_closed"] = df_closed_long["complaints_closed"].astype(int)df_closed_long.columns = ["firm_name", "firm_group", "reporting_period",                          "period", "product_category", "complaints_closed"]print(f"Total rows: {len(df_closed_long)}")df_closed_long.head()

## 4. Export Clean CSV FilesSave all three cleaned datasets. These CSVs are then imported into a PostgreSQL database where the analysis (8 SQL queries) is performed — see `queries/fca_complaints_queries.sql`.

In [ ]:
# Save all three clean datasets as CSV filesdf_long.to_csv("/kaggle/working/complaints_opened.csv", index=False)df_upheld_long.to_csv("/kaggle/working/complaints_upheld.csv", index=False)df_closed_long.to_csv("/kaggle/working/complaints_closed.csv", index=False)print("All three files saved:")print(f"  complaints_opened.csv  - {len(df_long)} rows")print(f"  complaints_upheld.csv  - {len(df_upheld_long)} rows")print(f"  complaints_closed.csv  - {len(df_closed_long)} rows")